# Reggression Model Training

To detect the features of the nodules we will train a regression model to predict the nodule features vector from a given image

## Imports
we need to first need to imports necessery packages

In [1]:


import torch
import lightning as L
import numpy as np
import pandas as pd
import pathlib
from pathlib import Path

from DetectionModel.src.models.regression_model import NoduleFeaturesModel
from DetectionModel.src.data_modules.regression_dataset_module import RegressionDataModule

from DetectionModel.constants.dataclasses.nodule_features import NoduleFeatures
from DetectionModel.constants.constants.regression_model import RegressionModelConstants
from DetectionModel.constants.constants.dataset import DatasetConstants
from common.constants import Metrics, ModelStage, Accelerator
from common.callbacks import CustomProgressBar
import lightning.pytorch.callbacks as LightningCallbacks
from common.callbacks.notification_callback import NtfyCallback
from lightning.pytorch.loggers import TensorBoardLogger
torch.serialization.add_safe_globals([pathlib.WindowsPath,pathlib.PosixPath])

d:\FinalsProject\MachineLearning\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Model and dataset preparation

since we need 1:1 alignment for patients, all the info is stored in a csv containing info for both the reggresion model and detection model, we'll pass the path to the csv to the lightning data module and set up the model,data module and trainer

In [2]:
reg_df_path = DatasetConstants.DATASET_DIR

DATASET_ROOT = DatasetConstants.PROJECT_ROOT

reg_dm = RegressionDataModule(
    metadata_csv=reg_df_path,
    dataset_root=DATASET_ROOT
)

model = NoduleFeaturesModel(learning_rate=1e-4,
                            conv_layers_channels=(32,64,128,256),
                            dense_layers_channels=(256,128,64)
                            )


callbacks=[
    LightningCallbacks.EarlyStopping(
        monitor=Metrics.DEFAULT_LOSS.get_variant(ModelStage.VAL),
        patience=15,
    ),
    NtfyCallback(
        model_name=RegressionModelConstants.MODEL_NAME,
    ),
    LightningCallbacks.ModelCheckpoint(
        dirpath=RegressionModelConstants.CHECKPOINT_DIR,
        filename=RegressionModelConstants.BEST_MODEL_CHECKPOINT_NAME,
        monitor=Metrics.DEFAULT_LOSS.get_variant(ModelStage.VAL),
        mode="min",
        save_top_k=1,
    
    ),
    LightningCallbacks.OnExceptionCheckpoint(
        dirpath=RegressionModelConstants.CHECKPOINT_DIR,
        filename=RegressionModelConstants.EXCEPTION_CHECKPOINT_FILE_NAME
    ),

    LightningCallbacks.LearningRateMonitor(logging_interval='epoch')
]

trainer = L.Trainer(
    accelerator=Accelerator.AUTO,
    devices=1,
    max_epochs=100,
    callbacks=callbacks,
    logger=TensorBoardLogger(
        save_dir=RegressionModelConstants.LOG_DIR,
        name=RegressionModelConstants.MODEL_NAME.replace(" ", "_")
    ),
    log_every_n_steps=10
)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


# Model Training


### Checkpoint Locating

befor starting training we need to check if a checkpoint of the model exists so we could start from where we left

In [3]:

checkpoints = RegressionModelConstants.CHECKPOINT_DIR.glob("*.ckpt")
checkpoints = list(map(lambda path: path.stem,checkpoints))

best_ckpt_path = RegressionModelConstants.CHECKPOINT_DIR / f"{RegressionModelConstants.BEST_MODEL_CHECKPOINT_NAME}.ckpt"
checkpoint_init = str(best_ckpt_path) if RegressionModelConstants.BEST_MODEL_CHECKPOINT_NAME in checkpoints else None


### Model training run

now that we've located whether a checkpoint exists or not, we'll start training

In [4]:

trainer.fit(model=model,datamodule=reg_dm,ckpt_path=checkpoint_init)

d:\FinalsProject\MachineLearning\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\checkpoint_connector.py:130: `.fit(ckpt_path=None)` was called without a model. The last model of the previous `fit` call will be used. You can pass `fit(ckpt_path='best')` to use the best model or `fit(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
d:\FinalsProject\MachineLearning\.venv\Lib\site-packages\lightning\pytorch\trainer\connectors\checkpoint_connector.py:190: .fit(ckpt_path="last") is set, but there is no last checkpoint available. No checkpoint will be loaded. HINT: Set `ModelCheckpoint(..., save_last=True)`.


┏━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ feature_extractor   │ Sequential │  388 K │ train │     0 │
│ 1 │ regressor           │ Sequential │ 42.0 K │ train │     0 │
│ 2 │ loss_fn             │ MSELoss    │      0 │ train │     0 │
│ 3 │ model_stage_metrics │ ModuleDict │      0 │ train │     0 │
└───┴─────────────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 430 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 430 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 46                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

d:\FinalsProject\MachineLearning\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 
'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)